In [ ]:
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage

# 1. CREDENCIALES DIRECTAS
load_dotenv()
mi_token = ""

remitente_email = os.getenv("CONFIG_EMAIL_REMITENTE")
remitente_password = os.getenv("CONFIG_EMAIL_PASSWORD")

os.environ["OPENAI_API_KEY"] = mi_token
os.environ["OPENAI_BASE_URL"] = "https://models.inference.ai.azure.com"

# 2. FUNCIONES DE PYHTON PURAS (Sin decoradores @tool para evitar conflictos de esquemas)
def obtener_ofertas_texto() -> str:
    return (
        "1. Razer DeathAdder V3 (Con cable): Oferta a $45.000 (Antes $59.990).\n"
        "2. Logitech G305 Lightspeed: Oferta a $35.500 (Antes $49.990).\n"
        "3. Attack Shark X6 Ultralight: Oferta a $39.990 (Antes $55.000).\n"
        "4. Logitech G Pro X Superlight 2: Oferta a $139.990 (Antes $169.000).\n"
        "5. Razer Cobra (Con cable): Oferta a $28.990 (Antes $38.000)."
    )

def enviar_mail_real(destinatario: str, cuerpo: str) -> bool:
    if not remitente_email or not remitente_password:
        return False
    msg = MIMEMultipart()
    msg['From'] = remitente_email
    msg['To'] = destinatario
    msg['Subject'] = "🎮 Tus Ofertas Gamer Solicitadas - MeliExpert 🖱️"
    
    cuerpo_html = f"""
    <html>
        <body>
            <h2 style='color: #2b6cb0;'>¡Hola! Aquí tienes tus ofertas de mouses:</h2>
            <hr>
            <pre style='font-family: sans-serif; font-size: 14px;'>{cuerpo}</pre>
            <hr>
            <p style='color: #718096; font-size: 12px;'>Enviado de forma segura a través del script local de Python.</p>
        </body>
    </html>
    """
    msg.attach(MIMEText(cuerpo_html, 'html'))
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as servidor:
            servidor.login(remitente_email, remitente_password)
            servidor.sendmail(remitente_email, destinatario, msg.as_string())
        return True
    except Exception as e:
        print(f"\n[Error de SMTP]: {e}")
        return False

# 3. MODELO BASE SÓLO PARA SALUDOS NORMALES
llm = ChatOpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=mi_token,
    model="gpt-4o",
    temperature=0.1
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres MeliExpert, un asesor tecnológico amigable. Conversas cordialmente sobre periféricos de computadora."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | llm

# 4. ENRUTADOR SEGURO (Si hay palabras clave, responde Python. Si no, responde el LLM)
def ejecutar_agente_antifiltro(inputs):
    texto_usuario = inputs["input"].lower()
    
    # DETECCIÓN DE CORREO: Interceptamos antes de ir a Azure
    if "correo" in texto_usuario or "mail" in texto_usuario or "enviar" in texto_usuario:
        # Extraer dirección usando '@'
        palabras = texto_usuario.split()
        correo_detectado = next((p for p in palabras if "@" in p), None)
        
        if not correo_detectado:
            return AIMessage(content="🎮 ¡Claro! Escribe tu correo electrónico completo (ejemplo@gmail.com) para poder despacharte la lista de ofertas de inmediato. 📧")
        
        correo_detectado = correo_detectado.strip(".,¡!?()\"'")
        
        # Procesamos las ofertas y enviamos el mail mediante Python directamente
        ofertas = obtener_ofertas_texto()
        exito = enviar_mail_real(correo_detectado, ofertas)
        
        if exito:
            return AIMessage(content=f"¡Listo, Deiner! 🚀 Acabo de enviar el top 5 de ofertas de mouses gamer directamente a tu bandeja en **{correo_detectado}**. ¡Revisa tu bandeja de entrada o spam! 🎮📧")
        else:
            return AIMessage(content="❌ Tuve un problema técnico interno con el servidor SMTP al intentar despachar el correo. Revisa tus credenciales en el archivo `.env`.")

    # DETECCIÓN DE SOLO VER OFERTAS
    elif "oferta" in texto_usuario or "mouse" in texto_usuario or "raton" in texto_usuario:
        ofertas = obtener_ofertas_texto()
        return AIMessage(content=f"🎮 ¡Aquí tienes las mejores ofertas vigentes de mouses gamer para tu setup! 🖱️\n\n{ofertas}\n\n¿Quieres que te envíe esta lista a tu correo electrónico?")

    # Si es una conversación común, usamos el LLM de manera segura
    return chain.invoke(inputs)

# 5. ESTRUCTURA DE MEMORIA
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

agent_with_memory = RunnableWithMessageHistory(
    RunnableLambda(ejecutar_agente_antifiltro),
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# 6. INTERFAZ INTERACTIVA
def iniciar_chat():
    print("\n--- 🎮 MeliExpert: Agente Activo (Modo Blindado Antifiltros) ---")
    config = {"configurable": {"session_id": "sesion_blindada_final"}}

    while True:
        pregunta = input("Tú: ")
        if pregunta.lower() in ["salir", "exit", "chau"]:
            break
        
        print("🤖 MeliExpert: ", end="")
        try:
            response = agent_with_memory.invoke({"input": pregunta}, config=config)
            print(response.content + "\n")
        except Exception as e:
            print(f"\n❌ Error durante la ejecución: {e}\n")

if __name__ == "__main__":
    iniciar_chat()